# 30 · federation · which clinical traits do the studies share?

Reads only the three end-product files `data/run_artifacts/<GSE>/clinical_dictionary.csv`: trait
names, how many samples record each, and whether it is 0/1. No expression values and no patient
records are read.

A trait is proposed as **shared** when two or more studies record a variable of the same definition
under the same name (the names were aligned in notebooks 06, 16 and 27). The table marks the
definition differences that remain. **This mapping is a proposal for approval; notebook 31 (the
meta-analysis) is not run until it is approved.**

In [1]:
source("../src/paths.R")
studies <- c("GSE65391", "GSE232381", "GSE135779")
dict <- do.call(rbind, lapply(studies, function(g) read.csv(art(g, "clinical_dictionary.csv"))))
wide <- reshape(dict[, c("study", "trait", "recorded")], idvar = "trait", timevar = "study", direction = "wide")
names(wide) <- sub("recorded\\.", "", names(wide))
wide$studies <- rowSums(!is.na(wide[, studies]))
wide[order(-wide$studies, wide$trait), ]

,trait,GSE65391,GSE232381,GSE135779,studies
,<chr>,<int>,<int>,<int>,<dbl>
5,age,157,NA,33,2
7,c3,149,NA,32,2
8,c4,148,NA,32,2
6,female,157,NA,33,2
17,hydroxychloroquine,156,NA,33,2
12,lymphocyte_count,137,NA,32,2
16,mycophenolate,157,NA,33,2
3,nephritis_any,98,NA,33,2
4,nephritis_proliferative,98,NA,31,2


## Proposed mapping

| trait | GSE65391 (array, 157 SLE first visits) | GSE135779 (pseudobulk, 33 SLE) | GSE232381 (bulk, 16 LN) | note |
|---|---|---|---|---|
| sledai, stage | yes | yes | no | same index; stage uses the same cut points |
| nephritis_any | nephritis class other than NoLN | Neph_all | no | GSE65391 records a class for 98 of 157 |
| nephritis_proliferative | Prolif or Proli+Membr | Neph_class III or IV | no | derived differently: class text vs class number |
| nephritis activity | no | no | nephritis_active | only GSE232381; not the same as nephritis_any |
| age, female | yes | yes | no | |
| c3, c4 | yes | yes | no | medians 96 / 14.5 and 87.5 / 12.0: same units assumed |
| wbc, neutrophil_count, lymphocyte_count, platelet_count | yes | yes | no | medians agree in scale (for example neutrophils 3.77 and 3.44) |
| oral_steroids, mycophenolate, hydroxychloroquine | yes | yes | no | |
| iv_steroids, cyclophosphamide, ds_dna_log10 | yes | no | no | GSE135779 records anti-dsDNA as a titer |
| methotrexate, mdg | no | yes | no | GSE135779 mdg is the authors' own 0–3 scale |
| sle_vs_healthy | not tested | yes | no | GSE65391 modules were tested within SLE only |

**What this means for the meta-analysis.** Every shared clinical trait is shared by **two** studies,
GSE65391 and GSE135779. GSE232381 records only nephritis activity, which no other study records, so
it can contribute module preservation (notebook 15) but not a pooled trait estimate.